# Form Factor Analysis

This notebook studies one concrete overcomplete-Wannier form factor in the uniform topological phase. It first computes the untruncated lower-band form factor for the chosen trial orbital in momentum space, then applies the same finite-`nshell` real-space truncation used in the project code and examines the resulting effective momentum-space form factor.

## Setup

Bootstrap imports from the repository `src/` tree and load the numerical stack.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
FIGDIR = ROOT / "form_factor_analysis" / "figs"
FIGDIR.mkdir(parents=True, exist_ok=True)

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["image.origin"] = "lower"
plt.rcParams["axes.grid"] = False

## Parameters

Choose a uniform Chern-insulator point, a trial orbital, and a small set of `nshell` values to compare.

In [ ]:
Nx = 50
Ny = 50
alpha = 1.0
trial_orbitals = "X"
center = (0, 0)
nshell_values = [0, 0.5, 1, 2, 4, 8]

print(f"Nx={Nx}, Ny={Ny}, alpha={alpha}, trial_orbitals={trial_orbitals}, center={center}")
print(f"nshell values: {nshell_values}")

## Helpers

These helpers build the exact two-band Bloch data, define the finite-`nshell` truncation as an explicit real-space mask `g(r)`, and then evaluate the effective single-band form factor using the convolution formula from the note.

In [ ]:
def trial_spinor(label):
    if label == "X":
        return np.array([1.0, 1.0], dtype=np.complex128) / np.sqrt(2.0)
    if label == "Y":
        return np.array([1.0, 1.0j], dtype=np.complex128) / np.sqrt(2.0)
    if label == "Z":
        return np.array([1.0, 0.0], dtype=np.complex128)
    raise ValueError(f"Unsupported trial orbital: {label!r}")


def nshell_label(nshell):
    if np.isclose(nshell, 0.5):
        return "1/2"
    if float(nshell).is_integer():
        return str(int(nshell))
    return str(nshell)


def support_offsets(nshell):
    if np.isclose(nshell, 0.5):
        return [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]
    if not float(nshell).is_integer():
        raise ValueError(f"Unsupported nshell value: {nshell!r}")
    n = int(round(nshell))
    return [(rx, ry) for rx in range(-n, n + 1) for ry in range(-n, n + 1)]


def truncation_mask(nshell, Rx=0, Ry=0):
    x = np.arange(Nx)[:, None]
    y = np.arange(Ny)[None, :]
    dx = ((x - Rx + Nx // 2) % Nx) - Nx // 2
    dy = ((y - Ry + Ny // 2) % Ny) - Ny // 2
    mask = np.zeros((Nx, Ny), dtype=bool)
    for rx, ry in support_offsets(nshell):
        mask |= (dx == rx) & (dy == ry)
    return mask


def shifted(arr):
    return np.fft.fftshift(arr, axes=(0, 1))


def k_axes():
    kx = 2 * np.pi * np.fft.fftfreq(Nx)
    ky = 2 * np.pi * np.fft.fftfreq(Ny)
    return np.fft.fftshift(kx / np.pi), np.fft.fftshift(ky / np.pi)


def k_in_pi(idx, kx_vals, ky_vals):
    ix, iy = idx
    return float(kx_vals[ix] / np.pi), float(ky_vals[iy] / np.pi)


def build_band_data(alpha, trial_orbitals):
    kx_vals = 2 * np.pi * np.fft.fftfreq(Nx)
    ky_vals = 2 * np.pi * np.fft.fftfreq(Ny)
    KX, KY = np.meshgrid(kx_vals, ky_vals, indexing="ij")
    nx = np.sin(KX)
    ny = np.sin(KY)
    nz = alpha - np.cos(KX) - np.cos(KY)
    nmag = np.sqrt(nx ** 2 + ny ** 2 + nz ** 2)
    nmag = np.where(nmag == 0.0, 1e-15, nmag)
    sx = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.complex128)
    sy = np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=np.complex128)
    sz = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)
    I2 = np.eye(2, dtype=np.complex128)
    hk = (
        nx[..., None, None] * sx
        + ny[..., None, None] * sy
        + nz[..., None, None] * sz
    ) / nmag[..., None, None]
    p_minus = 0.5 * (I2 - hk)
    eigvals, eigvecs = np.linalg.eigh(hk)
    u_minus = eigvecs[..., :, 0]
    tau = trial_spinor(trial_orbitals)
    proj_tau = np.einsum("...ab,b->...a", p_minus, tau, optimize=True)
    f_exact = np.einsum("...a,a->...", np.conjugate(u_minus), tau, optimize=True)
    return {
        "kx_vals": kx_vals,
        "ky_vals": ky_vals,
        "hk": hk,
        "p_minus": p_minus,
        "u_minus": u_minus,
        "tau": tau,
        "proj_tau": proj_tau,
        "f_exact": f_exact,
    }


def effective_form_factor_from_mask(mask, f_exact, u_minus, center=(0, 0)):
    g_tilde = np.fft.fft2(mask.astype(np.complex128))
    N = Nx * Ny
    kx_vals = 2 * np.pi * np.fft.fftfreq(Nx)
    ky_vals = 2 * np.pi * np.fft.fftfreq(Ny)
    ix_all = np.arange(Nx)
    iy_all = np.arange(Ny)
    f_eff = np.zeros((Nx, Ny), dtype=np.complex128)
    for ixp in range(Nx):
        qix = (ixp - ix_all) % Nx
        qx_vals = kx_vals[qix]
        for iyp in range(Ny):
            qiy = (iyp - iy_all) % Ny
            qy_vals = ky_vals[qiy]
            kernel = g_tilde[np.ix_(qix, qiy)]
            overlap = np.einsum("a,xya->xy", np.conjugate(u_minus[ixp, iyp]), u_minus, optimize=True)
            phase = np.exp(-1.0j * (qx_vals[:, None] * center[0] + qy_vals[None, :] * center[1]))
            f_eff[ixp, iyp] = np.sum(kernel * phase * f_exact * overlap) / N
    return f_eff, g_tilde


def plot_heatmap(ax, arr, title, cmap="viridis", colorbar_label=None):
    kx_plot, ky_plot = k_axes()
    im = ax.imshow(
        shifted(arr),
        extent=[ky_plot[0], ky_plot[-1], kx_plot[0], kx_plot[-1]],
        aspect="auto",
        cmap=cmap,
    )
    ax.set_title(title)
    ax.set_xlabel(r"$k_y / \pi$")
    ax.set_ylabel(r"$k_x / \pi$")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if colorbar_label is not None:
        cb.set_label(colorbar_label)
    return im

## Exact Form Factor

Here the exact lower-band form factor is defined directly from the Bloch eigenvector and the chosen trial spinor,

$$f(k) = \langle u_-(k) \mid \tau \rangle,$$

and the plotted spinor components come from the exact projected trial state `P_-(k) tau`.

For the special choice `alpha = 1` and `trial_orbitals = "X"`, the continuum zero is at `k / pi = (0.5, 0)`. For generic trial orbitals, the notebook reports the sampled minimum and avoids assuming a one-dimensional real zero.

In [ ]:
band = build_band_data(alpha=alpha, trial_orbitals=trial_orbitals)
u_minus = band["u_minus"]
proj_tau = band["proj_tau"]
f_exact = band["f_exact"]
kx_vals = band["kx_vals"]
ky_vals = band["ky_vals"]
f_exact_abs = np.abs(f_exact)

k_zero = np.unravel_index(np.argmin(f_exact_abs), f_exact_abs.shape)
kx_min, ky_min = k_in_pi(k_zero, kx_vals, ky_vals)
analytic_zero_k_over_pi = (0.5, 0.0) if (trial_orbitals == "X" and np.isclose(alpha, 1.0)) else None
print("Minimum of |f_exact| occurs at index:", k_zero)
print(rf"Minimum of |f_exact| occurs at k/pi = ({kx_min:.6f}, {ky_min:.6f})")
print("Minimum |f_exact| =", float(f_exact_abs[k_zero]))
if analytic_zero_k_over_pi is not None:
    print(rf"Analytic continuum zero for this case: k/pi = ({analytic_zero_k_over_pi[0]:.6f}, {analytic_zero_k_over_pi[1]:.6f})")
    print("Nearest sampled grid point misses kx/pi = 0.5 by", float(abs(kx_min - analytic_zero_k_over_pi[0])))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
plot_heatmap(axes[0], f_exact_abs, r"Exact $|f(k)|$", colorbar_label=r"$|f|$")
plot_heatmap(axes[1], np.abs(proj_tau[..., 0]), r"Exact $|(P_-\tau)_0(k)|$", colorbar_label=r"$|(P_-\tau)_0|$")
plot_heatmap(axes[2], np.abs(proj_tau[..., 1]), r"Exact $|(P_-\tau)_1(k)|$", colorbar_label=r"$|(P_-\tau)_1|$")
fig.savefig(FIGDIR / "exact_form_factor_components.png", dpi=180, bbox_inches="tight")
plt.show()

## Finite-`nshell` Real-Space Window

The truncation is represented directly as a periodic real-space support around the Wannier center. Integer `nshell` values keep the usual square window, while the special `nshell = 1/2` case keeps only the origin and nearest-neighbor unit cells. This is the real-space cutoff whose momentum-space effect we want to inspect.

In [ ]:
mask_show = [0, 0.5, 1, 2, 4]
fig, axes = plt.subplots(1, len(mask_show), figsize=(3.2 * len(mask_show), 3.2), constrained_layout=True)
if len(mask_show) == 1:
    axes = [axes]

for ax, ns in zip(axes, mask_show):
    mask = truncation_mask(ns, Rx=center[0], Ry=center[1]).astype(float)
    ax.imshow(np.fft.fftshift(mask))
    ax.set_title(f"nshell = {nshell_label(ns)}")
    ax.set_xlabel("y")
    ax.set_ylabel("x")

fig.savefig(FIGDIR / "nshell_real_space_masks.png", dpi=180, bbox_inches="tight")
plt.show()

## Effective Form Factor After Truncation

For each finite `nshell`, define the truncation as an explicit real-space mask. Integer `nshell` values keep the usual square support `|dx|, |dy| \le nshell`, while the special `nshell = 1/2` mask keeps only the origin and nearest-neighbor unit cells `(0,0)`, `(\pm 1,0)`, and `(0, \pm 1)`.

and evaluate the effective form factor from the note,

$$f_{\mathrm{eff}}(k') = \frac{1}{N}\sum_k \widetilde g(k'-k) e^{-i (k'-k)\cdot R} f(k)\,\langle u_-(k') \mid u_-(k) \rangle,$$

with `R = (0, 0)` in this notebook.

In [ ]:
results = {}
for ns in nshell_values:
    mask = truncation_mask(ns, Rx=center[0], Ry=center[1])
    f_eff, g_tilde = effective_form_factor_from_mask(mask, f_exact, u_minus, center=center)
    results[ns] = {
        "mask": mask,
        "g_tilde": g_tilde,
        "f_eff": f_eff,
        "f_eff_abs": np.abs(f_eff),
    }

for ns in nshell_values:
    k_min_idx = np.unravel_index(np.argmin(results[ns]["f_eff_abs"]), results[ns]["f_eff_abs"].shape)
    kx_min, ky_min = k_in_pi(k_min_idx, kx_vals, ky_vals)
    print(
        f"nshell={nshell_label(ns):>3}: min|f_eff|={results[ns]['f_eff_abs'].min():.6e}, "
        f"max|f_eff|={results[ns]['f_eff_abs'].max():.6e}, "
        f"k_min/pi=({kx_min:.6f}, {ky_min:.6f})"
    )

## `k`-Space Comparison

The first figure shows the full Brillouin-zone dependence of `|f_eff(k)|` for several `nshell` values. The second figure takes a `k_y = 0` cut, with momentum shown in units of `pi`, so it is easier to track how the mask broadens and reshapes the exact form factor.
When the special `alpha = 1`, `X`-trial analytic zero is available, the vertical dashed line marks it. For other trial choices, the cuts are shown without imposing that special-case zero.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
axes = axes.ravel()

plot_heatmap(axes[0], f_exact_abs, r"Exact $|f(k)|$", colorbar_label=r"$|f|$")
for ax, ns in zip(axes[1:], nshell_values):
    plot_heatmap(ax, results[ns]["f_eff_abs"], rf"Effective $|f(k)|$, nshell = {nshell_label(ns)}", colorbar_label=r"$|f_{\mathrm{eff}}|$")
for ax in axes[1 + len(nshell_values):]:
    ax.axis("off")

fig.savefig(FIGDIR / "effective_form_factor_maps.png", dpi=180, bbox_inches="tight")
plt.show()

ky = 2 * np.pi * np.fft.fftfreq(Ny)
ky0_index = int(np.argmin(np.abs(ky)))
kx_plot = 2 * np.pi * np.fft.fftfreq(Nx) / np.pi
sort_idx = np.argsort(kx_plot)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].plot(kx_plot[sort_idx], f_exact_abs[:, ky0_index][sort_idx], lw=2.5, label="exact")
for ns in nshell_values:
    axes[0].plot(kx_plot[sort_idx], results[ns]["f_eff_abs"][:, ky0_index][sort_idx], marker="o", ms=3, lw=1.3, label=f"nshell={nshell_label(ns)}")
axes[0].set_xlabel(r"$k_x / \pi$ at $k_y=0$")
axes[0].set_ylabel(r"$|f(k_x, 0)|$")
axes[0].set_title(r"Amplitude Cut of the Effective Form Factor")
if analytic_zero_k_over_pi is not None:
    axes[0].axvline(analytic_zero_k_over_pi[0], color="black", ls="--", lw=1.0, alpha=0.6, label="exact continuum zero")
axes[0].legend(loc="best")

for ns in nshell_values:
    axes[1].plot(kx_plot[sort_idx], np.angle(results[ns]["f_eff"][:, ky0_index][sort_idx]), marker="o", ms=3, lw=1.3, label=f"nshell={nshell_label(ns)}")
axes[1].set_xlabel(r"$k_x / \pi$ at $k_y=0$")
axes[1].set_ylabel(r"$\arg f_{\rm eff}(k_x, 0)$")
axes[1].set_title(r"Phase Cut Induced by Finite Truncation")
if analytic_zero_k_over_pi is not None:
    axes[1].axvline(analytic_zero_k_over_pi[0], color="black", ls="--", lw=1.0, alpha=0.6, label="exact continuum zero")
axes[1].legend(loc="best")

fig.savefig(FIGDIR / "effective_form_factor_cuts.png", dpi=180, bbox_inches="tight")
plt.show()

## Continuum-`k` Form Factor And Surface-Integral Chern Number

For a finite truncation support, the effective projected trial state is

$$|\psi_n(k)\rangle = \sum_{r \in \mathrm{supp}(g_n)} e^{-i k \cdot r} C_r |\tau\rangle, \qquad C_r = \int_{BZ} \frac{d^2q}{(2\pi)^2} e^{i q \cdot r} P_-(q).$$

The scalar form factor is still useful for visualizing zeros, but the Chern number below is computed from the normalized two-component state $|\psi_n(k)\rangle / \|\psi_n(k)\|$ over the full Brillouin zone. This replaces the previous local phase-winding proxy with a gauge-invariant discretized surface integral of Berry curvature.


In [ ]:
def dirichlet_kernel_1d(q, nshell):
    return 1 + 2 * sum(np.cos(m * q) for m in range(1, nshell + 1))


for ns in [0, 0.5, 1, 2, 4]:
    if np.isclose(ns, 0.5):
        print("g_{1/2}(kx, ky) = 1 + 2*cos(kx) + 2*cos(ky)")
    else:
        n = int(round(ns))
        terms = ["1"] + [f"2*cos({m} q)" for m in range(1, n + 1)]
        print(f"D_{n}(q) = {' + '.join(terms)}")


sx = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.complex128)
sy = np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=np.complex128)
sz = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)
I2 = np.eye(2, dtype=np.complex128)


def projector_at(kx, ky):
    kx, ky = np.broadcast_arrays(np.asarray(kx, dtype=float), np.asarray(ky, dtype=float))
    nx = np.sin(kx)
    ny = np.sin(ky)
    nz = alpha - np.cos(kx) - np.cos(ky)
    nmag = np.sqrt(nx ** 2 + ny ** 2 + nz ** 2)
    nmag = np.where(nmag == 0.0, 1e-15, nmag)
    hhat = (
        nx[..., None, None] * sx
        + ny[..., None, None] * sy
        + nz[..., None, None] * sz
    ) / nmag[..., None, None]
    return 0.5 * (I2 - hhat)


def projector_grid(grid_N):
    k = 2 * np.pi * (np.arange(grid_N) / grid_N - 0.5)
    KX, KY = np.meshgrid(k, k, indexing="ij")
    return k, k, projector_at(KX, KY)


def projector_fourier_coefficients(nshell, grid_N=181):
    kx_vals, ky_vals, p_minus = projector_grid(grid_N)
    kx = kx_vals[:, None]
    ky = ky_vals[None, :]
    coeffs = {}
    for rx, ry in support_offsets(nshell):
        phase_x = np.exp(1.0j * kx * rx)
        phase = phase_x * np.exp(1.0j * ky * ry)
        coeffs[(rx, ry)] = np.mean(phase[..., None, None] * p_minus, axis=(0, 1))
    return coeffs


def normalize_spinor_field(psi, eps=1e-12):
    norm = np.linalg.norm(psi, axis=-1)
    safe_norm = np.where(norm > eps, norm, 1.0)
    return psi / safe_norm[..., None], norm


def local_lower_spinor(kx, ky, refs=None, eps=1e-12):
    if refs is None:
        refs = [
            trial_spinor(trial_orbitals),
            np.array([1.0, 0.0], dtype=np.complex128),
            np.array([0.0, 1.0], dtype=np.complex128),
            np.array([1.0, 1.0], dtype=np.complex128) / np.sqrt(2.0),
            np.array([1.0, 1.0j], dtype=np.complex128) / np.sqrt(2.0),
        ]
    p_minus = projector_at(kx, ky)
    vec = np.einsum("...ab,b->...a", p_minus, refs[0], optimize=True)
    norm = np.linalg.norm(vec, axis=-1)
    for ref in refs[1:]:
        bad = norm < eps
        if not np.any(bad):
            break
        alt = np.einsum("...ab,b->...a", p_minus, ref, optimize=True)
        alt_norm = np.linalg.norm(alt, axis=-1)
        use_alt = bad & (alt_norm > norm)
        vec[use_alt] = alt[use_alt]
        norm[use_alt] = alt_norm[use_alt]
    return vec / np.where(norm > eps, norm, 1.0)[..., None]


def projected_trial_from_coeffs(kx, ky, coeffs):
    kx, ky = np.broadcast_arrays(np.asarray(kx, dtype=float), np.asarray(ky, dtype=float))
    tau = trial_spinor(trial_orbitals)
    projected = np.zeros(kx.shape + (2,), dtype=np.complex128)
    for (rx, ry), coeff in coeffs.items():
        phase = np.exp(-1.0j * (kx * rx + ky * ry))
        projected += phase[..., None] * (coeff @ tau)
    return projected


def f_eff_ky0_continuum(kx, coeffs, ky=0.0):
    kx = np.asarray(kx, dtype=float)
    ky_arr = np.full_like(kx, ky, dtype=float)
    projected = projected_trial_from_coeffs(kx, ky_arr, coeffs)
    u_minus = local_lower_spinor(kx, ky_arr)
    return np.einsum("...a,...a->...", np.conjugate(u_minus), projected, optimize=True)


def fhs_chern_number(spinors, eps=1e-12):
    psi, norm = normalize_spinor_field(spinors, eps=eps)
    overlap_x = np.einsum("...a,...a->...", np.conjugate(psi), np.roll(psi, -1, axis=0), optimize=True)
    overlap_y = np.einsum("...a,...a->...", np.conjugate(psi), np.roll(psi, -1, axis=1), optimize=True)
    Ux = np.where(np.abs(overlap_x) > eps, overlap_x / np.abs(overlap_x), 1.0 + 0.0j)
    Uy = np.where(np.abs(overlap_y) > eps, overlap_y / np.abs(overlap_y), 1.0 + 0.0j)
    plaquette = Ux * np.roll(Uy, -1, axis=0) * np.conjugate(np.roll(Ux, -1, axis=1)) * np.conjugate(Uy)
    berry_flux = np.angle(plaquette)
    return float(np.sum(berry_flux) / (2.0 * np.pi)), berry_flux, norm


def plaquette_axes_in_pi(grid_N):
    k_pi = 2.0 * (np.arange(grid_N) / grid_N - 0.5)
    return k_pi, k_pi


def flux_hotspot_summary(flux, kx_pi, ky_pi):
    idx = np.unravel_index(np.argmax(np.abs(flux)), flux.shape)
    return {
        "index": idx,
        "kx_pi": float(kx_pi[idx[0]]),
        "ky_pi": float(ky_pi[idx[1]]),
        "peak": float(flux[idx] / (2.0 * np.pi)),
        "peak_abs": float(np.abs(flux[idx]) / (2.0 * np.pi)),
    }


def plot_flux_panel(ax, flux, title, kx_pi, ky_pi, cmap="RdBu_r"):
    hotspot = flux_hotspot_summary(flux, kx_pi, ky_pi)
    vmax = hotspot["peak_abs"]
    vmax = max(vmax, 1e-12)
    im = ax.imshow(
        flux / (2.0 * np.pi),
        extent=[ky_pi[0], ky_pi[-1], kx_pi[0], kx_pi[-1]],
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=-vmax,
        vmax=vmax,
    )
    ax.plot(hotspot["ky_pi"], hotspot["kx_pi"], marker="o", ms=4, mec="black", mfc="none", mew=0.8)
    ax.set_title(title)
    ax.set_xlabel(r"$k_y / \pi$")
    ax.set_ylabel(r"$k_x / \pi$")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(r"plaquette Berry flux / $2\pi$")
    return hotspot


CHERN_GRID_N = 181
kx_chern = 2 * np.pi * (np.arange(CHERN_GRID_N) / CHERN_GRID_N - 0.5)
ky_chern = 2 * np.pi * (np.arange(CHERN_GRID_N) / CHERN_GRID_N - 0.5)
KX_CHERN, KY_CHERN = np.meshgrid(kx_chern, ky_chern, indexing="ij")
kx_plaquette_pi, ky_plaquette_pi = plaquette_axes_in_pi(CHERN_GRID_N)

exact_spinors = local_lower_spinor(KX_CHERN, KY_CHERN)
exact_chern, exact_flux, exact_norm = fhs_chern_number(exact_spinors)
print(f"exact lower band: C_surface={exact_chern:.8f}, rounded={int(np.rint(exact_chern)):+d}, min_norm={exact_norm.min():.3e}")
exact_hotspot = flux_hotspot_summary(exact_flux, kx_plaquette_pi, ky_plaquette_pi)
print(
    f"exact flux hotspot: index={exact_hotspot['index']}, "
    f"k/pi=({exact_hotspot['kx_pi']:.3f}, {exact_hotspot['ky_pi']:.3f}), "
    f"peak={exact_hotspot['peak']:.3e}"
)

chern_results = {}
for ns in nshell_values:
    coeffs = projector_fourier_coefficients(ns, grid_N=CHERN_GRID_N)
    psi_eff = projected_trial_from_coeffs(KX_CHERN, KY_CHERN, coeffs)
    chern, berry_flux, psi_norm = fhs_chern_number(psi_eff)
    f_cut = f_eff_ky0_continuum(kx_chern, coeffs)
    hotspot = flux_hotspot_summary(berry_flux, kx_plaquette_pi, ky_plaquette_pi)
    chern_results[ns] = {
        "coeffs": coeffs,
        "spinor": psi_eff,
        "berry_flux": berry_flux,
        "chern": chern,
        "min_norm": float(np.min(psi_norm)),
        "f_eff_ky0": f_cut,
        "hotspot": hotspot,
    }
    print(
        f"nshell={nshell_label(ns):>3}: C_surface={chern:.8f}, rounded={int(np.rint(chern)):+d}, "
        f"min|psi_eff|={np.min(psi_norm):.3e}, min|f_eff(kx,0)|={np.min(np.abs(f_cut)):.3e}"
    )
    print(
        f"    hotspot: index={hotspot['index']}, "
        f"k/pi=({hotspot['kx_pi']:.3f}, {hotspot['ky_pi']:.3f}), "
        f"peak={hotspot['peak']:.3e}"
    )


sort_chern = np.argsort(kx_chern / np.pi)
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for ns in nshell_values:
    vals = np.abs(chern_results[ns]["f_eff_ky0"])
    ax.plot(kx_chern[sort_chern] / np.pi, vals[sort_chern], lw=1.8, label=f"nshell={nshell_label(ns)}")
ax.set_xlabel(r"$k_x / \pi$ at $k_y=0$")
ax.set_ylabel(r"$|f_{\mathrm{eff}}(k_x, 0)|$")
ax.set_title(f"Generic $k_y=0$ Effective Form Factor Cut, trial={trial_orbitals}")
ax.legend(loc="best")
fig.savefig(FIGDIR / "continuum_generic_ky0_cuts.png", dpi=180, bbox_inches="tight")
plt.show()


plot_items = [("exact", exact_flux, exact_hotspot)] + [(f"nshell={nshell_label(ns)}", chern_results[ns]["berry_flux"], chern_results[ns]["hotspot"]) for ns in nshell_values]
ncols = 3
nrows = int(np.ceil(len(plot_items) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.8 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, (label, flux, hotspot) in zip(axes, plot_items):
    title = label + "\n" + rf"peak={hotspot['peak']:.2e} at $(k_x, k_y)/\pi=({hotspot['kx_pi']:.2f}, {hotspot['ky_pi']:.2f})$"
    plot_flux_panel(ax, flux, title, kx_plaquette_pi, ky_plaquette_pi)
for ax in axes[len(plot_items):]:
    ax.axis("off")
fig.savefig(FIGDIR / "chern_surface_integral_flux_maps.png", dpi=180, bbox_inches="tight")
plt.show()


The printed `C_surface` values are the Fukui-Hatsugai-Suzuki discretization of the Berry-curvature surface integral over the Brillouin zone. They should be close to integers when the normalized effective state is smooth and nonzero everywhere.

The diagnostic `min|psi_eff|` matters: if it is near machine precision, the effective state has a zero or near-zero on the integration grid, and the Chern number of that normalized field is not numerically stable. This is distinct from merely finding a zero of the scalar form factor on a one-dimensional cut.


## Interpretation Notes

- The notebook now treats finite `nshell` as a literal real-space mask `g(r)`, with a special cross-shaped nearest-neighbor support for `nshell = 1/2`.
- The effective form factor is computed from the single-band convolution formula, not from a separately rebuilt truncated OW basis.
- All reported and plotted momenta are shown in units of `pi`.
- The final Chern calculation uses a full-Brillouin-zone Berry-curvature surface integral of the normalized effective spinor, not a phase winding around a scalar zero.